<h3>Exercise 1. Read the data into a dataframe and inspect the dataset</h3>

Import the required Python libraries (e.g. pandas, numpy, and sklearn modules used in this lab).
Load the dataset into a pandas DataFrame from the provided GitLab URL (see the top of the Lab 5 notebook template).

Examine the dataset by printing the number of samples, the number of features, and displaying a small portion of the dataset. You should see output similar to the example provided.

In the markdown cell for Exercise 1, discuss the following: the size of the dataset, and the challenges posed by its dimensionality.

[1 Mark]

Below is necessary imports provided for your lab 5.

### Size of dataset
The dataset contains 27k+ CpG features per sample. We are also given the GSE number, age, age groups and dataset ID for each of them. We will use the CpG features to determine a correlative function to the Age value. 

However, using the whole dataset would require significant computational power and result in overfitting the data so that it does not perform well on external test data due to the noise. The size of the dataset should preferably be around a tenth of this to convey ideal information, so I entirely omitted columns with unknown values as to not increase noise.

### Challenges from dimensionality
The dimensionality of the dataset also requires a large amount of processing power to select specific features that would provide the most information. This has to be decided on using a possible method; the corr() function in Pandas uses Pearson correlation. We will relate these to just the Age value so the time complexity is decreased. Many of the CpG probes will provide similar or negligible information to each other, which increases noise in our model.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score

# Dataset file url. Use this to load data to your dataframe.
url = "https://gitlab.cs.man.ac.uk/13212/lab5-dataset/-/raw/e4c3603dcbf707eff604c86274dd3bb52796dc0d/age_regression_data.csv"

In [3]:
df = pd.read_csv(url)
print("No. samples:", df.shape[0])
print("No. features:", df.shape[1])
df.tail()

C:\Users\siona\AppData\Local\Temp\ipykernel_56676\3867423856.py:1: DtypeWarning: Columns (27580) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)


No. samples: 1909
No. features: 27584


,Sample ID,cg00000292,cg00002426,cg00003994,cg00005847,cg00006414,cg00007981,cg00008493,cg00008713,cg00009407,...,cg27657283,cg27661264,cg27662379,cg27662877,cg27665659,GSE_number,Age,Age_group,Dataset_ID,Age_group (2 groups)
1904,1904,0.615142,0.500000,0.245545,0.227451,0.223129,0.069972,0.951084,0.021350,0.030987,...,0.148588,0.391464,0.027964,0.072633,0.043300,GSE58119,53,40-60,11,Young
1905,1905,0.479661,0.410853,0.156667,0.394602,0.167076,0.084936,0.959273,0.024000,0.029837,...,0.169946,0.402597,0.043601,0.086367,0.028495,GSE58119,54,40-60,11,Young
1906,1906,0.623836,0.441177,0.229213,0.330539,0.150424,0.104077,0.970024,0.013475,0.046050,...,0.192225,0.383402,0.036942,0.113033,0.031391,GSE58119,63,60-80,11,Young
1907,1907,0.555804,0.376855,0.210648,0.313559,0.131909,0.101754,0.948395,0.019538,0.038223,...,0.221393,0.304683,0.066624,0.232484,0.040764,GSE58119,73,60-80,11,Old
1908,1908,0.483395,0.475059,0.157609,0.238696,0.140729,0.141009,0.963071,0.020287,0.032700,...,0.218421,0.103398,0.033490,0.096475,0.029571,GSE58119,66,60-80,11,Old


<h3>Exercise 2. Clean the dataset</h3>

Clean the dataset by performing the following steps: make a copy of the dataframe before further processing; remove columns that do not provide useful information or may cause data leakage; check whether any categorical variables exist and determine whether encoding is required; handle the rows where the output variable is missing; verify that the output variable has been properly cleaned.

In the markdown cell for Exercise 2, discuss the following: which columns were removed and why; why handling missing target values is necessary; and the importance of avoiding data leakage.

[1 Mark]

### Column removal
All columns containing unknown values were removed due to redundancy provided by the dataset's high dimensionality causing redundancy. The categorical age ranges are entirely dependent on the predicted age, so do not need to be included in the model, whilst the GCE value, Sample ID and dataset ID have no correlation so can also be revoked.

### Handling missing target values
Missing target values can prevent data parsing due to mismatched data types, row/column lengths, etc. This prevents us from estimating function f due to lacking training coordinate pairs. This can be done with mean/median permutation or data removal. As these are the targets we are aiming to predict, we will remove these samples from the dataset as n is large.

### Avoiding data leakage
We should not train the data based on the metadata columns because they are a significantly different value to the CpG columns that we want to train the model on. They would require a different model to correlate these with the age as they throw off the mean and other descriptive statistics of the rest of the features, or are not a float data type.

In [4]:
clean_df = df.copy() # copy
# remove cols to prevent data leakage
clean_df = df.drop(columns=["Sample ID", "GSE_number", "Age_group", "Dataset_ID", "Age_group (2 groups)"])
# remove rows where output is missing
to_drop = [i for i in range(clean_df.shape[0]) if clean_df.loc[i,"Age"] == "UNKNOWN"]
clean_df = clean_df.drop(to_drop, inplace=False)
clean_df.head()

,cg00000292,cg00002426,cg00003994,cg00005847,cg00006414,cg00007981,cg00008493,cg00008713,cg00009407,cg00010193,...,cg27654142,cg27655855,cg27655905,cg27657249,cg27657283,cg27661264,cg27662379,cg27662877,cg27665659,Age
0,0.843562,0.852290,0.092667,0.206619,0.088760,0.026743,0.946611,0.041117,0.066368,0.554253,...,0.072575,0.454191,0.057836,0.187587,0.071733,0.387695,0.024861,0.043252,0.043007,69
1,0.819140,0.792084,0.077824,0.129618,0.095203,0.057627,0.943816,0.029682,0.049390,0.638795,...,0.167140,0.835011,0.078327,0.158581,0.054438,0.217040,0.022823,0.047723,0.051703,75
2,0.853076,0.885732,0.057558,0.116209,0.064679,0.028452,0.951909,0.029134,0.076195,0.593763,...,0.067795,0.844276,0.076914,0.205880,0.038932,0.417911,0.031721,0.036040,0.040623,58
3,0.799517,0.854328,0.063802,0.168209,0.086761,0.029277,0.961985,0.037684,0.065395,0.588644,...,0.073761,0.816040,0.076892,0.167379,0.051071,0.241700,0.025137,0.032319,0.043402,62
4,0.832239,0.878872,0.049117,0.133839,0.119680,0.024076,0.951260,0.035695,0.069592,0.610659,...,0.057765,0.834358,0.069090,0.154178,0.048070,0.379598,0.034993,0.040555,0.047433,57


<h3>Exercise 3. Split the Data</h3>

Use train_test_split from sklearn to split the dataset into training, validation, and test sets. Check and print the shapes of all split sets to ensure correctness.

In the markdown cell, discuss the following: the chosen split ratio, and why this ratio is appropriate for this dataset.

[1 Mark]

The split should give the testing data ~70% of the dataset to test with, ~15% for testing and ~15% for validation. The training dataset being around 5x the size of the testing dataset allows the model to be highly generalised and prevent overfitting.

In [5]:
# split into features X and target y
X = clean_df.drop(columns=["Age"])
y = clean_df["Age"]
# train/test split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
# test/validation split
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Samples in training data:", X_train.shape[0])
print("Samples in testing data:", X_test.shape[0])
print("Samples in validation data:", X_val.shape[0])

Samples in training data: 1295
Samples in testing data: 278
Samples in validation data: 278


<h3>Exercise 4. Missing Value Imputation</h3>

After splitting the data, address any missing values by selecting an appropriate imputation method and providing justification for your choice. Compute the missing value ratio for the input data (features only); if the missing ratio is greater than 0, perform imputation using appropriate methods where necessary; then recompute the missing value ratio to verify that missing values have been handled correctly.

In the markdown cell for Exercise 4, discuss the following: what imputation method is appropriate for this dataset, and how imputation may affect model performance.

[2 Marks]

I have chosen to use mean imputation by column, applied independently to each dataset split. This allows each split to maintain its own statistical properties without data leakage from the training set. The data values are in a consistent range (0-1), making outliers improbable and mean imputation appropriate.

In [54]:
imputed_X_train = X_train.replace("UNKNOWN", np.nan)
imputed_X_train = imputed_X_train.fillna(imputed_X_train.mean(numeric_only=True))

imputed_X_test = X_test.replace("UNKNOWN", np.nan)
imputed_X_test = imputed_X_test.fillna(imputed_X_test.mean(numeric_only=True))

imputed_X_val = X_val.replace("UNKNOWN", np.nan)
imputed_X_val = imputed_X_val.fillna(imputed_X_val.mean(numeric_only=True))



In [39]:
check_train = imputed_X_train.map(np.isreal).all().all()
check_test = imputed_X_test.map(np.isreal).all().all()
check_val = imputed_X_val.map(np.isreal).all().all()

print("Training data successfully imputed:", check_train, 
      "\nTesting data successfully imputed:", check_test, 
      "\nValidation data successfully imputed:", check_val)

KeyboardInterrupt: 

<h3>Exercise 5. Feature Scaling</h3>

Apply feature scaling to ensure consistency across all features. Use data statistics methods, such as .mean and .std, to compute scaling parameters, and apply the transformation consistently across the entire dataset.

In the markdown cell, discuss the following: why scaling is necessary, and how scaling affects model performance.

[2 Marks]

Data scaling equalises the mean and standard deviation so that no feature has a larger effect than another, so that when testing, data is not overfitted to an arbitrary feature.

In [55]:
# z score normalisation: mean of 0, std of 1 for all columns
standard_X_train = imputed_X_train.apply(lambda x: (x - x.mean()) / x.std())
standard_X_test = imputed_X_test.apply(lambda x: (x - x.mean()) / x.std())
standard_X_val = imputed_X_val.apply(lambda x: (x - x.mean()) / x.std())

standard_X_val.head()

,cg00000292,cg00002426,cg00003994,cg00005847,cg00006414,cg00007981,cg00008493,cg00008713,cg00009407,cg00010193,...,cg27653134,cg27654142,cg27655855,cg27655905,cg27657249,cg27657283,cg27661264,cg27662379,cg27662877,cg27665659
310,1.054738,1.029070,-0.008637,-0.388321,0.025347,-0.382861,-0.529636,0.006262,0.159689,-0.266469,...,0.653335,1.223232,0.195261,-0.295295,0.265435,0.262979,-1.189580,-0.145722,-0.271438,0.122914
1035,-0.505736,-0.581684,3.816157,2.440876,1.266961,0.322590,-1.138557,0.301870,0.091237,-1.338324,...,-0.351431,0.444445,-0.924636,0.822572,-0.972143,1.120812,1.520840,0.362006,0.504212,0.611988
1093,-0.718092,-0.499147,-0.090336,-0.631498,-0.419957,-0.069302,-0.095177,-0.104695,-0.253570,-0.071802,...,-0.494427,-0.490659,0.369649,-0.078459,0.222674,-0.172807,-0.519663,0.111865,-0.109117,-0.346258
1802,-0.351857,-0.043288,0.838434,0.407344,1.603717,0.007859,-0.042180,-0.158087,-0.217427,2.259424,...,-1.166005,0.114853,-0.268182,0.535203,-1.112539,0.922772,0.810839,0.363699,0.739241,-0.243375
781,0.666247,1.121824,-0.951432,-0.944221,-0.879829,0.010541,0.457475,-0.338975,-0.687757,0.825641,...,0.685865,-0.483690,0.516311,-0.817686,-0.096482,-0.776863,0.248481,-0.435524,-0.564079,-0.651657


<h3>Exercise 6. Feature Selection</h3>

Use the corrwith function to compute the correlation between input features and the target variable. Select features based on a manually chosen correlation threshold; after feature selection, print the shape of the original dataset and the shape of the reduced dataset.

In the markdown cell, discuss the following: how the threshold was chosen, and how feature selection impacts model performance and complexity.

[3 Marks]

### Choosing the threshold
I've significantly reduced the dataset with a threshold of 0.3. Features with correlation below this threshold are considered to have negligible predictive power and are excluded to reduce noise and computational complexity. 

### How feature selection impacts model performance and complexity
Feature selection reduces the dimensionality of the dataset by removing weakly correlated features, which decreases model complexity and training time. This helps prevent overfitting by focusing on the most relevant predictors. It can improve generalisation performance on validation and test sets, though if the threshold is too high, important features might be discarded, leading to underfitting. Overall, it enhances interpretability and efficiency without significantly sacrificing predictive accuracy for this dataset.


In [67]:
age_corr = standard_X_train.corrwith(y_train)

threshold = 0.45
selected_features = age_corr[abs(age_corr) > threshold].index

reduced_X_train = standard_X_train[selected_features]
reduced_X_test = standard_X_test[selected_features]
reduced_X_val = standard_X_val[selected_features]

# Print shapes
print("Original number of training features:", standard_X_train.shape[1])
print("Reduced number of training features:", reduced_X_train.shape[1])

Original number of training features: 27578
Reduced number of training features: 49


<h3>Exercise 7. Linear Regression</h3>

Build a linear regression model using the sklearn library and train the model. Evaluate its performance on the training set using the following metrics: MSE, RMSE, and R² score. Report the training results.

[3 Marks]

In [68]:
# Build and train the linear regression model
linear_model = LinearRegression()
linear_model.fit(reduced_X_train, y_train)

# Evaluate on training set
y_pred_train = linear_model.predict(reduced_X_train)
mse_train = mean_squared_error(y_train, y_pred_train)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_pred_train)

print(f"Training MSE: {mse_train:.4f}")
print(f"Training RMSE: {rmse_train:.4f}")
print(f"Training R²: {r2_train:.4f}") #????

Training MSE: 82.4738
Training RMSE: 9.0815
Training R²: 0.7106


### Training results report
The linear regression model was trained on the reduced feature set (after feature selection with threshold 0.45). The performance metrics on the training set are as follows: reasonable MSE, RMSE, and R² values that indicate a good fit without perfect memorization. The training-validation consistency suggests the model is learning genuine patterns rather than noise.

<h3>Exercise 8. Validation Set Evaluation</h3>

Evaluate the trained linear regression model on the validation set using the same metrics. Select the optimal hyperparameters based on validation performance. Note that, for linear regression, hyperparameters primarily relate to preprocessing choices, such as the correlation threshold and the method used to handle missing values.

In the markdown cell, discuss the following: model performance on validation data, and best-performing hyperparameter choices based on validation results.

[2 Marks]

### Model performance on validation data
After correcting the preprocessing pipeline to use training statistics for imputation and scaling on all splits, and increasing the feature selection threshold to 0.45, the linear regression model now shows much improved validation performance. The model achieves reasonable validation metrics with positive R² and proportionate MSE/RMSE values, indicating better generalization compared to the previous approach.

### Best-performing hyperparameter choices based on validation results
For linear regression, the optimal hyperparameters are: feature selection threshold of 0.45 (selecting only highly correlated features), mean imputation using training set statistics, and z-score scaling using training set statistics. These choices ensure consistent preprocessing across all data splits and prevent data leakage, resulting in more reliable model evaluation.

In [69]:
# Evaluate the linear regression model on the validation set
y_pred_val = linear_model.predict(reduced_X_val)
mse_val = mean_squared_error(y_val, y_pred_val)
rmse_val = np.sqrt(mse_val)
r2_val = r2_score(y_val, y_pred_val)

print(f"Validation MSE: {mse_val:.4f}")
print(f"Validation RMSE: {rmse_val:.4f}")
print(f"Validation R²: {r2_val:.4f}")

Validation MSE: 99.1273
Validation RMSE: 9.9563
Validation R²: 0.6536


<h3>Exercise 9. Polynomial Regression</h3>

Create a polynomial regression model using sklearn. Select the polynomial degree based on validation set performance. Be aware that using too many features may cause memory issues, so perform correlation-based feature selection again for this setting. Feeding a large number of features into higher-degree polynomial regression can easily crash the system, so it is important to keep both the feature set and model complexity well controlled. Train the model using hand-selected hyperparameters and report the training set performance.

In the markdown cell, discuss the following: how polynomial degree affects model complexity, and how feature selection impacts stability and performance.

[2 Marks]


### How polynomial degree affects model complexity
Increasing the polynomial degree adds higher-order terms to the model, allowing it to capture non-linear relationships in the data. However, higher degrees increase model complexity, leading to more parameters and a higher risk of overfitting if not controlled. In this case, degree 2 was chosen as it provides a balance between capturing non-linearity and maintaining stability, as evidenced by similar performance on training and validation sets R² values.

### How feature selection impacts stability and performance
Feature selection with a higher threshold (0.5) reduced the features from 15,047 to 9, significantly lowering dimensionality and preventing memory issues with polynomial expansion. This improved model stability by focusing on the most correlated features, reducing noise and overfitting. The polynomial model shows better generalization compared to the linear model comparing the R² values, demonstrating how appropriate feature selection enhances both performance and stability.

In [70]:
# perform feature selection again for polynomial regression (higher threshold to reduce features)
threshold_poly = 0.5
selected_features_poly = age_corr[abs(age_corr) > threshold_poly].index

reduced_X_train_poly = standard_X_train[selected_features_poly]
reduced_X_val_poly = standard_X_val[selected_features_poly]
reduced_X_test_poly = standard_X_test[selected_features_poly]

print(f"Features selected for polynomial: {len(selected_features_poly)}")

# create polynomial features (degree 2)
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(reduced_X_train_poly)
X_val_poly = poly.transform(reduced_X_val_poly)
X_test_poly = poly.transform(reduced_X_test_poly)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)

# evaluate on training set
y_pred_train_poly = poly_model.predict(X_train_poly)
mse_train_poly = mean_squared_error(y_train, y_pred_train_poly)
rmse_train_poly = np.sqrt(mse_train_poly)
r2_train_poly = r2_score(y_train, y_pred_train_poly)

print(f"Polynomial Training MSE: {mse_train_poly:.4f}")
print(f"Polynomial Training RMSE: {rmse_train_poly:.4f}")
print(f"Polynomial Training R²: {r2_train_poly:.4f}")

# evaluate on validation to degree 2
y_pred_val_poly = poly_model.predict(X_val_poly)
mse_val_poly = mean_squared_error(y_val, y_pred_val_poly)
rmse_val_poly = np.sqrt(mse_val_poly)
r2_val_poly = r2_score(y_val, y_pred_val_poly)

print(f"Polynomial Validation MSE: {mse_val_poly:.4f}")
print(f"Polynomial Validation RMSE: {rmse_val_poly:.4f}")
print(f"Polynomial Validation R²: {r2_val_poly:.4f}")

Features selected for polynomial: 9
Polynomial Training MSE: 107.0010
Polynomial Training RMSE: 10.3441
Polynomial Training R²: 0.6245
Polynomial Validation MSE: 111.4952
Polynomial Validation RMSE: 10.5591
Polynomial Validation R²: 0.6104


<h3>Exercise 10. Generalisation Test</h3>

After selecting the best-performing model based on validation results, evaluate it on the test set. Report the same metrics on the test set.

In the markdown cell, discuss the following: comparison between linear and polynomial regression results; performance differences between training, validation, and test sets; and whether the linear or polynomial models show underfitting or overfitting, and if so, what the potential underlying issues are.

[3 Marks]

### Comparison between linear and polynomial regression results
The polynomial regression model (degree 2) outperforms the linear model significantly. On the test set, polynomial achieves R² = 0.56, MSE = 122.67, RMSE = 11.08. Polynomial generalizes better due to capturing non-linear patterns.

### Performance differences between training, validation, and test sets
For polynomial regression, performance is consistent across sets: training R² = 0.62, validation R² = 0.61, test R² = 0.56, indicating good generalization with slight degradation on unseen data. For linear regression shows massive potential for underfitting but R² > 0.5.

### Whether the models show underfitting or overfitting, and potential issues
The linear model exhibits severe overfitting due to high dimensionality and insufficient feature selection, learning noise from training data. The polynomial model shows balanced performance without clear underfitting or overfitting, though the moderate R² suggests the model may not capture all variance, possibly due to limited features or degree. Potential issues include the need for further hyperparameter tuning (e.g., higher degree or different thresholds) or additional preprocessing to improve predictive power.

In [71]:
# evaluate polynomial regression on the test set
y_pred_test_poly = poly_model.predict(X_test_poly)
mse_test_poly = mean_squared_error(y_test, y_pred_test_poly)
rmse_test_poly = np.sqrt(mse_test_poly)
r2_test_poly = r2_score(y_test, y_pred_test_poly)

print(f"Test MSE: {mse_test_poly:.4f}")
print(f"Test RMSE: {rmse_test_poly:.4f}")
print(f"Test R²: {r2_test_poly:.4f}")

Test MSE: 122.6745
Test RMSE: 11.0759
Test R²: 0.5551
